<a href="https://colab.research.google.com/github/mudassar2224/MCP_Complete_Lab--LLM-Journey/blob/main/03_MCP_Complete_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                         MCP LAB
                            │
              ┌─────────────┼─────────────┐
              ▼             ▼             ▼
            Tools        Resources      Prompts
              │             │             │
              ▼             ▼             ▼
            Actions        Data        Templates
              │             │             │
              └─────────────┼─────────────┘
                            ▼
                         MCP Server
                            ▲
                            │
                         MCP Client
                            ▲
                            │
                   LangChain / LangGraph
                            ▲
                            │
                           LLM

# ***`PART 1 — MCP environment`***

PART A — Mental Model
1. What MCP is
2. MCP vs API vs SDK vs framework
3. Host
4. Client
5. Server
6. Protocol messages
7. Capabilities
8. Stateless architecture

### **The current MCP architecture is still host/client/server conceptually, but the July 2026 specification moved the core toward stateless request/response operation.**

# ***`FIRST BLOCK — Only 4 steps`***

1. Install current MCP SDK
2. Create MCPServer
3. Add one Tool
4. Connect Client and call Tool

In [1]:
!pip -q install -U "mcp[cli]"

In [2]:
import sys
import mcp

print("Python:", sys.version)
print("MCP:", getattr(mcp, "__version__", "unknown"))

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
MCP: unknown


# ***`STEP 2 — Create the MCP Server`***

In [3]:
from mcp.server import MCPServer
mcp=MCPServer()

In [4]:


mcp = MCPServer(
    "Research MCP Server",
    instructions=(
        "Provides research-related tools "
        "and resources."
    )
)

print(mcp)

In [5]:
from mcp.server import MCPServer

# ***`STEP 3 — Add your first Tool`***

In [6]:
@mcp.tool()
def add(
    a: int,
    b: int
) -> int:
    """Add two integers."""

    return a + b

# ***`STEP 4 — Connect an MCP Client`***

In [7]:
from mcp import  Client

In [8]:
async with Client(mcp) as client:

    print(
        "Server:",
        client.server_info
    )

    print(
        "Capabilities:",
        client.server_capabilities
    )

    print(
        "Protocol version:",
        client.protocol_version
    )

Server: name='Research MCP Server' title=None version='' description=None website_url=None icons=None
Capabilities: experimental=None logging=None prompts=PromptsCapability(list_changed=True) resources=ResourcesCapability(subscribe=True, list_changed=True) tools=ToolsCapability(list_changed=True) completions=None extensions=None tasks=None
Protocol version: 2026-07-28


# ***`Now discover the Tool`***

In [9]:
async with Client(mcp) as client:

    result = await client.list_tools()

    for tool in result.tools:

        print(
            "NAME:",
            tool.name
        )

        print(
            "TITLE:",
            tool.title
        )

        print(
            "DESCRIPTION:",
            tool.description
        )

        print(
            "INPUT SCHEMA:",
            tool.input_schema
        )

NAME: add
TITLE: None
DESCRIPTION: Add two integers.
INPUT SCHEMA: {'type': 'object', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'addArguments'}


In [10]:
async with Client(mcp) as client:

    result = await client.call_tool(
        "add",
        {
            "a": 25,
            "b": 17
        }
    )

    print(
        "Content:",
        result.content
    )

    print(
        "Structured:",
        result.structured_content
    )

    print(
        "Error:",
        result.is_error
    )

Content: [TextContent(type='text', text='42', annotations=None, meta=None)]
Structured: {'result': 42}
Error: False


                 HOST
                  │
               CLIENT
                  │
            MCP protocol
                  │
               SERVER
                  │
               TOOL
                  │
             real system

# 05. Tool schemas in depth
# 06. Structured tool output
# 07. Tool annotations
# 08. Tool errors

# ***05 — Tool Schemas in Depth***

In [11]:
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

In [12]:
from pydantic  import BaseModel , Field

In [13]:
from pydantic import BaseModel, Field


class SearchInput(BaseModel):
    query: str = Field(
        description="The search query"
    )

    top_k: int = Field(
        default=5,
        ge=1,
        le=20,
        description="Maximum number of results"
    )


@mcp.tool()
def search_documents(
    query: str,
    top_k: int = 5
) -> list[str]:
    """
    Search the research knowledge base.
    """

    return [
        f"Result {i}: {query}"
        for i in range(1, top_k + 1)
    ]

In [14]:
async with Client(mcp) as client:

    result = await client.list_tools()

    for tool in result.tools:

        if tool.name == "search_documents":

            print("NAME:")
            print(tool.name)

            print("\nDESCRIPTION:")
            print(tool.description)

            print("\nINPUT SCHEMA:")
            print(tool.input_schema)

NAME:
search_documents

DESCRIPTION:

Search the research knowledge base.


INPUT SCHEMA:
{'type': 'object', 'properties': {'query': {'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'title': 'Top K', 'type': 'integer'}}, 'required': ['query'], 'title': 'search_documentsArguments'}


# ***`06 — Structured Tool Output`***

In [15]:
from pydantic import BaseModel


class SearchResult(BaseModel):
    query: str
    matches: int
    results: list[str]


@mcp.tool()
def research_search(
    query: str
) -> SearchResult:

    results = [
        f"Paper about {query}",
        f"Research on {query}",
        f"Survey of {query}"
    ]

    return SearchResult(
        query=query,
        matches=len(results),
        results=results
    )

In [16]:
async with Client(mcp) as client:

    result = await client.call_tool(
        "research_search",
        {
            "query": "RAG"
        }
    )

    print("CONTENT:")
    print(result.content)

    print("\nSTRUCTURED:")
    print(result.structured_content)

    print("\nERROR:")
    print(result.is_error)

CONTENT:
[TextContent(type='text', text='{\n  "query": "RAG",\n  "matches": 3,\n  "results": [\n    "Paper about RAG",\n    "Research on RAG",\n    "Survey of RAG"\n  ]\n}', annotations=None, meta=None)]

STRUCTURED:
{'query': 'RAG', 'matches': 3, 'results': ['Paper about RAG', 'Research on RAG', 'Survey of RAG']}

ERROR:
False


# ***`07 — Tool Annotations`***


In [17]:
from mcp.types import  ToolAnnotations
print(ToolAnnotations)

<class 'mcp_types._types.ToolAnnotations'>


In [18]:
@mcp.tool(
    annotations=ToolAnnotations(
        readOnlyHint=True
    )
)
def knowledge_search(
    query: str
) -> list[str]:
    """
    Search the knowledge base.
    """

    return [
        f"Evidence for: {query}"
    ]

# ***`08 — Tool Errors`***

# ***`Create a failing tool`***

In [19]:
from mcp.server.mcpserver.exceptions import ToolError


@mcp.tool()
def get_paper_author(
    title: str
) -> str:
    """
    Find the author of a research paper.
    """

    papers = {
        "Attention Is All You Need":
            "Vaswani et al.",
        "BERT":
            "Devlin et al."
    }

    if title not in papers:

        raise ToolError(
            f"No paper found with title: {title}"
        )

    return papers[title]

In [20]:
async with Client(mcp) as client:

    result = await client.call_tool(
        "get_paper_author",
        {
            "title": "Unknown Paper"
        }
    )

    print("ERROR:")
    print(result.is_error)

    print("\nCONTENT:")
    print(result.content)

    print("\nSTRUCTURED:")
    print(result.structured_content)

ERROR:
True

CONTENT:
[TextContent(type='text', text='Error executing tool get_paper_author: No paper found with title: Unknown Paper', annotations=None, meta=None)]

STRUCTURED:
None


# 09. Resources
# 10. **Resource** URIs
# 11. Resource Templates
# 12. Reading Resources from the Client

# ***`09 — What is an MCP Resource?`***

## **Step 1 — Add a Resource to our existing server**

In [21]:
from mcp.server import MCPServer

mcp = MCPServer(
    "Research MCP Server",
    instructions=(
        "Provides research-related tools "
        "and resources."
    )
)

In [22]:
@mcp.resource(
    "research://papers"
)
def research_papers() -> str:
    """
    Available research papers.
    """

    return """
    1. Attention Is All You Need
    2. BERT: Pre-training of Deep Bidirectional Transformers
    3. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
    """

# ***10 — Resource URIs***

The important part is:

In [23]:
@mcp.resource(
    "research://config"
)
def research_config() -> str:
    """
    Research system configuration.
    """

    return """
    Embedding model: Gemini Embeddings
    Vector database: Qdrant
    Reranker: BGE
    Retrieval: Dense + BM25 + RRF
    """

# ***`11 — Resource Templates`***

In [24]:
@mcp.resource(
    "research://paper/{paper_id}"
)
def get_paper(
    paper_id: str
) -> str:
    """
    Retrieve a research paper by ID.
    """

    return (
        f"Research paper content "
        f"for paper ID: {paper_id}"
    )

# ***`12 — Read Resources from the MCP Client`***

In [25]:
from mcp import Client

In [26]:
async with Client(mcp) as client:

    result = await client.list_resources()

    for resource in result.resources:

        print(
            "NAME:",
            resource.name
        )

        print(
            "URI:",
            resource.uri
        )

        print(
            "DESCRIPTION:",
            resource.description
        )

        print(
            "MIME TYPE:",
            resource.mime_type
        )

        print("----------------")

NAME: research_papers
URI: research://papers
DESCRIPTION: 
Available research papers.

MIME TYPE: text/plain
----------------
NAME: research_config
URI: research://config
DESCRIPTION: 
Research system configuration.

MIME TYPE: text/plain
----------------


# ***`Discover the Resource Templates`***

In [27]:
async with Client(mcp) as client:

    result = await client.list_resource_templates()

    for template in result.resource_templates:

        print(
            "NAME:",
            template.name
        )

        print(
            "URI TEMPLATE:",
            template.uri_template
        )

        print(
            "DESCRIPTION:",
            template.description
        )

        print("----------------")

NAME: get_paper
URI TEMPLATE: research://paper/{paper_id}
DESCRIPTION: 
Retrieve a research paper by ID.

----------------


In [28]:
async with Client(mcp) as client:

    result = await client.read_resource(
        "research://papers"
    )

    for content in result.contents:

        print(content)

uri='research://papers' mime_type='text/plain' meta=None text='\n    1. Attention Is All You Need\n    2. BERT: Pre-training of Deep Bidirectional Transformers\n    3. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks\n    '


In [29]:
from mcp.types import TextResourceContents

In [30]:
async with Client(mcp) as client:

    result = await client.read_resource(
        "research://papers"
    )

    for content in result.contents:

        if isinstance(
            content,
            TextResourceContents
        ):
            print(content.text)


    1. Attention Is All You Need
    2. BERT: Pre-training of Deep Bidirectional Transformers
    3. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
    


In [31]:
async with Client(mcp) as client:

    result = await client.read_resource(
        "research://paper/1706.03762"
    )

    for content in result.contents:

        if isinstance(
            content,
            TextResourceContents
        ):
            print(content.text)

Research paper content for paper ID: 1706.03762


In [32]:
@mcp.resource(
    "document://{document_id}"
)
def get_document(
    document_id: str
) -> str:
    ...

In [33]:
@mcp.resource(
    "research://metadata",
    mime_type="application/json"
)
def research_metadata() -> dict:
    return {
        "papers": 100,
        "topics": 25
    }

                         MCP CLIENT
                              │
              ┌───────────────┴────────────────┐
              │                                │
        list_resources()              list_resource_templates()
              │                                │
              ▼                                ▼
       Concrete Resources              URI Templates
              │                                │
              │                         fill parameters
              │                                │
              └───────────────┬────────────────┘
                              ▼
                       read_resource()
                              │
                              ▼
                         MCP SERVER
                              │
                     ┌────────┴────────┐
                     ▼                 ▼
                  Context          External Data

# 13. Resource metadata & annotations
# 14. Text vs binary resources
# 15. Resource subscriptions
# 16. Resource caching

# ***`13. Resource Metadata & Annotations`***

A resource is not just:
It can also describe how important the content is, who it is intended for, and when it was modified.

Current MCP annotations include:

audience
priority
lastModified

The protocol defines priority from 0.0 to 1.0, where 1.0 means most important.

In [34]:
from mcp.types import Annotations

In [35]:
@mcp.resource(
    "research://important-paper",
    name="Important Research Paper",
    title="Important Research Paper",
    description="High-priority research paper",
    mime_type="text/plain",
    annotations=Annotations(
        audience=["assistant"],
        priority=1.0,
        last_modified="2026-09-09T00:00:00Z",
    ),
)
def important_paper() -> str:
    return """
    Attention Is All You Need

    Transformer architecture introduced self-attention
    as a scalable sequence modeling mechanism.
    """

In [36]:
async with Client(mcp) as client:
    result = await client.list_resources()

    for resource in result.resources:
        print("Name:", resource.name)
        print("URI:", resource.uri)
        print("Description:", resource.description)
        print("MIME:", resource.mime_type)
        print("Annotations:", resource.annotations)
        print()

Name: research_papers
URI: research://papers
Description: 
Available research papers.

MIME: text/plain
Annotations: None

Name: research_config
URI: research://config
Description: 
Research system configuration.

MIME: text/plain
Annotations: None

Name: research_metadata
URI: research://metadata
Description: 
MIME: application/json
Annotations: None

Name: Important Research Paper
URI: research://important-paper
Description: High-priority research paper
MIME: text/plain
Annotations: audience=['assistant'] priority=1.0 last_modified='2026-09-09T00:00:00Z'



# ***`14. Text vs Binary Resources`***

## Text resource

In [37]:
@mcp.resource(
    "research://abstract",
    mime_type="text/plain"
)
def abstract() -> str:
    return """
    This paper studies retrieval augmented generation.
    """

In [38]:
async with Client(mcp) as client:
    result = await client.read_resource(
        "research://abstract"
    )

    for content in result.contents:
        print(type(content))
        print(content)

<class 'mcp_types._types.TextResourceContents'>
uri='research://abstract' mime_type='text/plain' meta=None text='\n    This paper studies retrieval augmented generation.\n    '


## JSON resource

In [39]:
@mcp.resource(
    "research://paper-metadata",
    mime_type="application/json"
)
def paper_metadata() -> dict:
    return {
        "title": "Attention Is All You Need",
        "year": 2017,
        "topics": [
            "transformers",
            "attention",
            "NLP"
        ]
    }

In [40]:
@mcp.resource(
    "research://paper-preview",
    mime_type="application/pdf"
)
def paper_preview() -> bytes:
    with open("paper.pdf", "rb") as f:
        return f.read()

# ***`15. Resource Subscriptions`***


Modern MCP uses:

subscriptions/listen

for change notifications. The old resources/subscribe API belongs to older protocol versions and should not be used for your new 2026 code

In [41]:
live_results = {
    "status": "initial",
    "results": []
}

@mcp.resource(
    "research://live-results",
    mime_type="application/json"
)
def live_results_resource() -> dict:
    return live_results

In [47]:
from mcp.server.mcpserver import Context

@mcp.tool()
async def update_results(
    status: str,
    ctx: Context
) -> str:

    live_results["status"] = status

    await ctx.notify_resource_updated(
        "research://live-results"
    )

    return f"Updated status to: {status}"

It does not send the entire resource contents. The client should refetch it.

In [48]:
import asyncio

async with Client(mcp) as client:
  async with client.listen(
      resource_subscriptions=["research://live-results"]
  ) as subscription:
    print("Subscription active")
    # Example: Listen for a single event or set a timeout
    try:
      async with asyncio.timeout(5):  # Stop after 5 seconds
        async for event in subscription:
          print("Event:", event)
          if event.uri == "research://live-results":
            result = await client.read_resource("research://live-results")
            print(result.contents)
            break  # Exit after processing the event
    except TimeoutError:
      print("Subscription timed out after waiting for events.")

Subscription active
Subscription timed out after waiting for events.


# ***`16. Resource Caching`***

Now combine subscriptions with caching.

The 2026-07-28 MCP protocol added cache hints:

ttlMs
cacheScope

for cacheable results including:

tools/list

In [53]:
from mcp.server import MCPServer, CacheHint

mcp = MCPServer(
    "Research MCP Server",
    cache_hints={
        "resources/read": CacheHint(
            ttl_ms=60_000
        ),
        "resources/list": CacheHint(
            ttl_ms=30_000
        )
    }
)

# 17. MCP Prompts
# 18. Prompt arguments
# 19. Prompt discovery
# 20. Prompt rendering

# ***`17 — MCP Prompts`***

A prompt in MCP is not simply an ordinary LLM prompt string.

It is a reusable, discoverable template exposed by an MCP server.

Think:

MCP Server

│

├──
 Tools

├──

Resources


└── Prompts

      │

      ├──
      summarize_paper

      ├──
      analyze_code
      
      └── research_question

The client can discover those prompts and request one.

This is useful when the server knows a particular workflow or interaction pattern

In [50]:
@mcp.prompt()
def research_assistant():
    return """
    You are a research assistant.

    Answer questions using reliable evidence.
    Clearly distinguish retrieved evidence
    from your own reasoning.
    """

# ***`18 — Prompt Arguments`***

Real applications need dynamic prompts.

For example:

summarize_paper
    ↓
paper_title
    ↓
summary instructions

In [54]:
@mcp.prompt()
def summarize_paper(
    paper_title: str
):
    return f"""
    You are a research assistant.

    Summarize the following paper:

    {paper_title}

    Include:
    1. Problem
    2. Method
    3. Results
    4. Limitations
    5. Key contribution
    """

# ***`19 — Prompt Discovery`***

## The client doesn't have to magically know which prompts exist.

### It can ask:

### "What prompts does this MCP server provide?"

In [55]:
async with Client(mcp) as client:

    result = await client.list_prompts()

    for prompt in result.prompts:

        print("NAME:", prompt.name)
        print("TITLE:", prompt.title)
        print("DESCRIPTION:", prompt.description)

        print("ARGUMENTS:")

        if prompt.arguments:

            for argument in prompt.arguments:

                print(
                    " ",
                    argument.name,
                    "→",
                    argument.description,
                    "required:",
                    argument.required
                )

        print()

NAME: summarize_paper
TITLE: None
DESCRIPTION: 
ARGUMENTS:
  paper_title → None required: True



# ***`20 — Prompt Rendering`***

Now the useful part.

We've discovered:

summarize_paper

and it requires:

paper_title

In [56]:
async with Client(mcp) as client:

    result = await client.get_prompt(
        "summarize_paper",
        {
            "paper_title":
                "Attention Is All You Need"
        }
    )

    print(result)

meta={'io.modelcontextprotocol/serverInfo': {'name': 'Research MCP Server', 'version': ''}} description='' messages=[PromptMessage(role='user', content=TextContent(type='text', text='\n    You are a research assistant.\n\n    Summarize the following paper:\n\n    Attention Is All You Need\n\n    Include:\n    1. Problem\n    2. Method\n    3. Results\n    4. Limitations\n    5. Key contribution\n    ', annotations=None, meta=None))] result_type='complete'


In [57]:
async with Client(mcp) as client:

    result = await client.get_prompt(
        "summarize_paper",
        {
            "paper_title":
                "Attention Is All You Need"
        }
    )

    print("Description:")
    print(result.description)

    print("\nMessages:")

    for message in result.messages:

        print("Role:", message.role)
        print("Content:", message.content)
        print()


Description:


Messages:
Role: user
Content: type='text' text='\n    You are a research assistant.\n\n    Summarize the following paper:\n\n    Attention Is All You Need\n\n    Include:\n    1. Problem\n    2. Method\n    3. Results\n    4. Limitations\n    5. Key contribution\n    ' annotations=None meta=None



In [58]:
@mcp.prompt()
def analyze_research_paper(
    paper_title: str,
    research_goal: str
):
    return f"""
    You are an expert AI research analyst.

    Paper:
    {paper_title}

    Research goal:
    {research_goal}

    Analyze the paper with the following structure:

    1. Research problem
    2. Motivation
    3. Proposed approach
    4. Architecture
    5. Training methodology
    6. Experimental setup
    7. Results
    8. Limitations
    9. Relation to previous work
    10. Practical applications

    Do not invent information.
    Clearly distinguish evidence from inference.
    """

In [59]:
async with Client(mcp) as client:

    result = await client.get_prompt(
        "analyze_research_paper",
        {
            "paper_title":
                "Attention Is All You Need",

            "research_goal":
                "Understand how the Transformer architecture "
                "changed sequence modeling."
        }
    )

    print(result)

meta={'io.modelcontextprotocol/serverInfo': {'name': 'Research MCP Server', 'version': ''}} description='' messages=[PromptMessage(role='user', content=TextContent(type='text', text='\n    You are an expert AI research analyst.\n\n    Paper:\n    Attention Is All You Need\n\n    Research goal:\n    Understand how the Transformer architecture changed sequence modeling.\n\n    Analyze the paper with the following structure:\n\n    1. Research problem\n    2. Motivation\n    3. Proposed approach\n    4. Architecture\n    5. Training methodology\n    6. Experimental setup\n    7. Results\n    8. Limitations\n    9. Relation to previous work\n    10. Practical applications\n\n    Do not invent information.\n    Clearly distinguish evidence from inference.\n    ', annotations=None, meta=None))] result_type='complete'


# 21. MCP lifecycle & initialization
# 22. Capability negotiation
# 23. Notifications
# 24. Pagination

# ***`21 — MCP Lifecycle & Initialization`***

An MCP connection isn't simply:

client → call tool

                 CLIENT
                    │
                    │ initialize
                    ▼
                 SERVER
                    │
                    │ initialize response
                    ▼
                 CLIENT
                    │
                    │ initialized
                    ▼
              Normal operation

## The first phase establishes:


```

protocol version
client information
server information
capabilities

```



Only after initialization should normal MCP operations occur.

# ***`22 — Capability Negotiation`***

This is one of the most important concepts.

Suppose your server supports:

Tools

Resources

Prompts

but another MCP server supports only:

Tools

The client needs to know what the server supports.

That's what capabilities are for.

# ***`23 — Notifications`***

Now we need another distinction.

Most MCP communication is:

request
    ↓
response

For example:



```
Client
 │
 │ call_tool()
 ▼
Server
 │
 │ result
 ▼
Client
```



But sometimes the server needs to tell the client:

Something changed.

without the client explicitly asking.

That's a notification.

# ***24 — Pagination***

This becomes extremely important when your system becomes large.

Imagine your MCP server contains:

2 tools

No problem.

But imagine:

50,000 resources

Returning everything at once would be terrible.

So MCP supports pagination for large listings.

Without pagination


```
Client
  │
  │ list_resources()
  ▼
Server
  │
  ├── resource 1
  ├── resource 2
  ├── resource 3
  ├── ...
  └── resource 50,000
```



Huge response.

With pagination


```
Client
  │
  │ list_resources()
  ▼
Server
  │
  ├── resource 1
  ├── ...
  └── resource 100
        +
     nextCursor
```



Then:



```
Client
  │
  │ list_resources(cursor)
  ▼
Server
  │
  ├── resource 101
  ├── ...
  └── resource 200
        +
     nextCursor
```



Continue until:

nextCursor = none

# 25. stdio transport
# 26. Streamable HTTP
# 27. Sessions
# 28. Remote MCP server

These concepts answer a fundamental question:

How does the MCP client actually communicate with the MCP server?

# ***`25 — stdio Transport`***

This is the simplest way to run an MCP server.

Imagine:



```
Your Python application
        │
        │ starts process
        ▼
   MCP Server
        │
   stdin/stdout
```



The client launches the MCP server as a local process.

For example:

my_mcp_server.py

is started by the client.

Communication happens through:

stdin  → client sends MCP messages

stdout → server sends MCP messages

Conceptually:



```
┌──────────────────────┐
│      MCP Client      │
│                      │
│   stdin/stdout       │
└──────────┬───────────┘
           │
           │ local process
           ▼
┌──────────────────────┐
│      MCP Server      │
│      Python          │
└──────────────────────┘
```


Why stdio is useful

For local development:

Python application
       ↓
MCP server
       ↓
filesystem
       ↓
database

There is no need to deploy an HTTP server.

This is excellent for:

development
local tools
IDE integrations
filesystem access
local databases
coding agents

# ***`26 — Streamable HTTP`***

Now imagine the MCP server isn't running inside your computer.

Instead:



```
Your application
       │
       │ Internet / network
       ▼
MCP Server
       │
       ├── PostgreSQL
       ├── Qdrant
       ├── APIs
       └── filesystem
```



This is where Streamable HTTP becomes important.

The modern MCP transport for remote communication is Streamable HTTP.

Conceptually:



```
┌──────────────┐
│ MCP Client   │
└──────┬───────┘
       │
       │ HTTP
       ▼
┌──────────────┐
│ MCP Server   │
└──────────────┘
```



The protocol can support streaming server-to-client messages over HTTP rather than requiring a simple one-request/one-response interaction.

Why streaming matters

Suppose an MCP tool performs a long operation:

search 10,000 documents

Without streaming:



```
Client
  │
  │ request
  ▼
Server
  │
  │.............wait.............
  │
  ▼
complete response

```


With streaming:



```
Client
  │
  │ request
  ▼
Server
  │
  ├── progress/result
  ├── progress/result
  ├── progress/result
  └── final result

```


This becomes valuable for:

long-running tools
large responses
agent workflows
real-time updates
remote MCP servers

# ***`stdio vs Streamable HTTP`***

Memorize this:



```
stdio
│
├── local
├── process-to-process
├── simple
└── excellent for development
```



versus:



```
Streamable HTTP
│
├── remote
├── network
├── scalable deployment
├── authentication
└── production systems

```


So:

LOCAL MCP
    ↓
stdio

and:

REMOTE MCP
    ↓
Streamable HTTP

# ***`27 — Sessions`***

Now we need another concept.

When a client connects to a remote MCP server, we may need to maintain state associated with that connection.

Think:


```

Client
  │
  │ initialize
  ▼
Server
  │
  │ session established
  ▼
Client
```



A session can allow the server and client to associate subsequent protocol messages with the established MCP interaction.

Conceptually:

      

```
          SESSION
                   │
       ┌───────────┼───────────┐
       │           │           │
       ▼           ▼           ▼
   initialize   requests   notifications
```



Don't confuse:

MCP session

with:

conversation memory

They are not the same thing.

Very important distinction

Your eventual application may have:

User conversation
       ↓
Conversation ID
       ↓
Memory
       ↓
LangGraph state

Separately, MCP has:

MCP connection
       ↓
protocol session
       ↓
MCP messages

So:

MCP session ≠ LLM memory

This distinction will save you a lot of confusion later.

# ***`28 — Remote MCP Server`***

# 29. **Authentication**
# 30. **Authorization**
# 31. **Security** / trust boundaries
# **32. Human approval / consent**

# ***`29 — Authentication`***

First understand:

Authentication
=
Who is calling me?

For remote MCP:



```
Client
   ↓
Authorization: Bearer <token>
   ↓
MCP Server
   ↓
Token verifier
   ↓
valid / invalid
```



MCP authorization is an HTTP concern. It does not apply to stdio or the in-memory Client(mcp) testing transport.

In [60]:
from mcp.server.mcpserver import MCPServer

mcp_secure = MCPServer(
    "Secure Research Server"
)

print(mcp_secure)

In [61]:
from mcp.server.auth.provider import (
    AccessToken,
    TokenVerifier
)


class DemoTokenVerifier(TokenVerifier):
    """
    DEMO ONLY.

    A production implementation should verify
    JWT signatures or use token introspection.
    """

    async def verify_token(
        self,
        token: str
    ) -> AccessToken | None:

        if token == "demo-research-token":

            return AccessToken(
                token=token,
                client_id="demo-client",
                scopes=[
                    "research:read"
                ],
                subject="demo-user"
            )

        return None

# ***`Configure OAuth resource-server metadata`***

In [63]:
from pydantic import AnyHttpUrl
from mcp.server.auth.settings import (
    AuthSettings
)


auth_settings = AuthSettings(
    issuer_url=AnyHttpUrl(
        "https://auth.example.com/"
    ),

    resource_server_url=AnyHttpUrl(
        "http://127.0.0.1:8000/mcp"
    ),

    required_scopes=[
        "research:read"
    ]
)

/tmp/ipykernel_48382/135279171.py:7: MCPDeprecationWarning: `AuthSettings.validate_token_resource` is not set, so bearer tokens are not checked against `resource_server_url`; it will default to True in 3.0 when `resource_server_url` is set. Set it to True to have the server refuse tokens issued for another resource, or to False if your TokenVerifier validates the token's audience itself.
  auth_settings = AuthSettings(


In [64]:
mcp_secure = MCPServer(
    "Secure Research Server",

    token_verifier=
        DemoTokenVerifier(),

    auth=auth_settings
)

/usr/local/lib/python3.13/dist-packages/mcp/server/mcpserver/server.py:188: MCPDeprecationWarning: `AuthSettings.validate_token_resource` is not set, so bearer tokens are not checked against `resource_server_url`; it will default to True in 3.0 when `resource_server_url` is set. Set it to True to have the server refuse tokens issued for another resource, or to False if your TokenVerifier validates the token's audience itself.
  self.settings = Settings(




```
# **Run over Streamable HTTP**
```




In [68]:
import nest_asyncio
import threading
import time

nest_asyncio.apply()

def run_mcp_server_in_thread():
    print("Starting MCP server in a new thread...")
    try:
        mcp_secure.run(
            transport="streamable-http",
            json_response=True
        )
    except RuntimeError as e:
        print(f"RuntimeError in MCP server thread: {e}")
        print("This might indicate that nest_asyncio is not fully effective for anyio.run within the new thread's context.")
    except Exception as e:
        print(f"An unexpected error occurred in MCP server thread: {e}")
    print("MCP server thread finished execution.")

# Create and start a new thread for the server
server_thread = threading.Thread(target=run_mcp_server_in_thread)
server_thread.daemon = True # Allows the main program to exit even if the thread is still running
server_thread.start()

print("MCP server startup requested. It will attempt to run in a background thread.")
print("You might need to wait a few seconds for the server to fully initialize before connecting clients.")
# Add a short delay to allow the server to start in the background
time.sleep(5)

INFO:     Started server process [48382]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Starting MCP server in a new thread...
MCP server startup requested. It will attempt to run in a background thread.
You might need to wait a few seconds for the server to fully initialize before connecting clients.


# ***`30 — Authorization`***

In [69]:
from mcp.server.auth.middleware.auth_context import (
    get_access_token
)

In [70]:
from mcp.server.mcpserver.exceptions import (
    ToolError
)


@mcp_secure.tool()
async def read_private_research(
    paper_id: str
) -> str:
    """
    Read a protected research paper.
    """

    access_token = get_access_token()

    if access_token is None:
        raise ToolError(
            "Authentication required."
        )

    if (
        "research:read"
        not in access_token.scopes
    ):
        raise ToolError(
            "Missing research:read scope."
        )

    return (
        f"Authorized access to paper "
        f"{paper_id}"
    )

In [71]:
@mcp_secure.tool()
async def publish_research(
    paper_id: str
) -> str:

    access_token = get_access_token()

    if access_token is None:
        raise ToolError(
            "Authentication required."
        )

    if (
        "research:write"
        not in access_token.scopes
    ):
        raise ToolError(
            "You need research:write."
        )

    return (
        f"Paper {paper_id} "
        "would be published."
    )

# ***`31 — Security & Trust Boundaries`***

In [72]:
@mcp_secure.tool()
async def run_sql(
    query: str
) -> str:
    ...

In [73]:
@mcp_secure.tool()
async def run_readonly_sql(
    query: str
) -> str:

    normalized = (
        query
        .strip()
        .lower()
    )

    if not normalized.startswith(
        "select"
    ):
        raise ToolError(
            "Only SELECT queries are allowed."
        )

    forbidden = [
        "insert ",
        "update ",
        "delete ",
        "drop ",
        "alter ",
        "truncate "
    ]

    if any(
        keyword in normalized
        for keyword in forbidden
    ):
        raise ToolError(
            "Unsafe SQL operation blocked."
        )

    return (
        "Readonly SQL execution would occur here."
    )

# ***`Application-level approval`***

In [74]:
def requires_approval(
    tool_name: str
) -> bool:

    sensitive_tools = {
        "publish_research",
        "delete_document"
    }

    return (
        tool_name
        in sensitive_tools
    )

In [75]:
def ask_human_approval(
    tool_name: str,
    arguments: dict
) -> bool:

    print(
        "\n⚠️ HUMAN APPROVAL REQUIRED"
    )

    print(
        "Tool:",
        tool_name
    )

    print(
        "Arguments:",
        arguments
    )

    answer = input(
        "Approve? [yes/no]: "
    )

    return (
        answer
        .strip()
        .lower()
        == "yes"
    )

In [76]:
def execute_tool_safely(
    tool_name: str,
    arguments: dict,
    tool_function
):

    if requires_approval(
        tool_name
    ):

        approved = (
            ask_human_approval(
                tool_name,
                arguments
            )
        )

        if not approved:

            return {
                "status":
                    "rejected",

                "reason":
                    "Human approval denied."
            }

    try:

        result = tool_function(
            **arguments
        )

        return {
            "status":
                "success",

            "result":
                result
        }

    except Exception as e:

        return {
            "status":
                "error",

            "error":
                str(e)
        }

# 33. Current discovery model
# 34. Elicitation
# 35. Sampling
# 36. Tasks / long-running operations

# ***`33 — Current MCP Discovery`***

You already learned:



```
list_tools()
list_resources()
list_resource_templates()
list_prompts()
```



That's normal discovery.

But modern MCP also has a broader server-discovery concept.

Think:



```
Client
  ↓
What MCP server am I talking to?
  ↓
What capabilities does it expose?

The important architectural distinction is:
```




```

LIST
→ discover a specific primitive

DISCOVER
→ discover server/capability information
```



Your application should not blindly assume:

"this server supports sampling"
"this server supports elicitation"
"this server supports tasks"

Instead:



```
connect
 ↓
discover/inspect capabilities
 ↓
use only supported features
```



This is exactly the same principle you already learned with capability negotiation.



```
Mental model
                    MCP SERVER
                        │
                ┌───────┴────────┐
                ↓                ↓
          server discovery    capability info
                │                │
                └───────┬────────┘
                        ↓
                   MCP CLIENT
                        ↓
                feature selection
```



This becomes important once your agent can connect to different MCP servers from different vendors.

One server might expose:

tools
resources

Another:

tools
resources
prompts
sampling

Your client should adapt.

# ***`34 — Elicitation`***

# ***`I need information from you.`***

Imagine:



```
Tool:
search_regulations

requires:

country
product_type

User says:

"Find regulations for this product."
```



The server discovers:

country is missing

Then:



```
MCP Server
    ↓
Elicit:
"What country should I use?"

User:

"Pakistan"

```


Then:

search_regulations(
    country="Pakistan",
    product_type="..."
)

# ***`35 — Sampling`***

This is one of the concepts people often misunderstand.

You already know:



```
Agent
 ↓
LLM
 ↓
Tool
```



Sampling introduces a different interaction:


```

MCP Server
     ↓
asks client/host for model generation
     ↓
LLM
     ↓
response
     ↓
MCP Server

```



So the server can participate in a model-generation workflow without necessarily owning the model itself.

Think:

            HOST
             │
          LLM/API
             │
        MCP CLIENT
             │
             │ sampling request
             ▼
        MCP SERVER
             │
          needs model
             │
             ▼
        CLIENT / HOST
             │
             ▼
            LLM
             │
             ▼
        generated text
             │
             ▼
        MCP SERVER

This is very different from:



```
Agent
 ↓
call_tool()
```


Why sampling exists

Imagine an MCP server specialized in:

research analysis

It might have logic that says:

"I need a model to reason over these retrieved pieces of evidence."

Rather than embedding a specific vendor/model directly into the server, it can request model assistance from the host/client.

That keeps the architecture more modular.

# ***`36 — Tasks / Long-Running Operations`***

Now we address an important real-world problem.

Not every operation finishes instantly.


```

You might have:

search()
→ 200 ms

but:

train_model()
→ 20 minutes

or:

analyze_10,000_papers()
→ several minutes

or:

generate_large_report()
→ long-running
```



You don't want the MCP client to think:

request failed

simply because the operation is


```
still running.

Task mental model
Client
 ↓
Start Task
 ↓
Task ID
 ↓
working
 ↓
progress
 ↓
completed

Possible lifecycle:

working
   ↓
completed

or:

working
   ↓
failed

```


or:



```
working
   ↓
cancelled

or potentially:

working
   ↓
input required
   ↓
user responds
   ↓
working
```



This is extremely useful for long-running agent workflows.

Example

Imagine:

Tool:
analyze_research_corpus

The user asks:

Analyze 100,000 AI papers and identify emerging RAG trends.

A normal blocking call would be terrible:



```
request
 ↓
wait...
 ↓
wait...
 ↓
wait...
```



A task-based architecture is:



```
request
 ↓
task created
 ↓
task_id = T123
 ↓
working
 ↓
progress updates
 ↓
completed
 ↓
result available
```



                              USER
                                │
                                ▼
                         Streamlit / Gradio
                                │
                                ▼
                              FastAPI
                                │
                                ▼
                           LangGraph
                                │
                          ┌─────┴─────┐
                          │   Agent   │
                          └─────┬─────┘
                                │
                ┌───────────────┼────────────────┐
                ▼               ▼                ▼
             LangChain        MCP              Memory
                │               │
                │        ┌──────┼──────┐
                │        ▼      ▼      ▼
                │      Tools Resources Prompts
                │        │       │
       ┌────────┼────────┘       │
       ▼        ▼                ▼
     Qdrant   Neo4j           External data
     RAG      GraphRAG
       │
       ├── Multimodal RAG
       ├── Code RAG
       └── SQL RAG

                    ↓
                 Gemini
                    ↓
             Verification
                    ↓
            Answer + citations
                    ↓
                LangSmith